# Variables

- ice velocity (phase-based)
- basal friction inversion
- Pressure melting point (Van Lieff.)

To do:
- Specularity (https://par.nsf.gov/biblio/10656793-radar-classification-antarctic-ice-bed-interface-version)
- distance from coast?!


In [14]:
import pandas as pd
import xarray as xr
import numpy as np
import torch

# from paths import path_to_tiles_metadata_EAGLE_2016
from paths import path_to_phase_based_ice_vel
from paths import path_to_beta_basal_friction

# from paths import path_to_tiles_tensor
from paths import path_to_tiles_metadata
from paths import path_to_tiles_metadata_enriched

## Load metadata

In [2]:
meta = pd.read_csv(path_to_tiles_metadata)
meta

,granule_id,flight,x_center_idx,y_center_idx,tile_id,X,Y,LON,LAT,bed_elevation,surface_elevation,ice_thickness,partial_bed_reflect,srf_reflect,tile_along_track_span_m,tile_vertical_span_m,bed_std_m,peak_bed_echo_db,campaign,survey
0,ER2HI1B_2017023_ICP8_JKB2r_F07T02a_000,ICP8/F07,168,1054,ER2HI1B_2017023_ICP8_JKB2r_F07T02a_000_0168,2.233971e+06,-1.003400e+06,114.187485,-67.733750,-364.43,774.85,1139.28,-51.14,-13.08,6386.525126,-567.550895,45.556204,196.545,ICP8,ER2HI
1,ER2HI1B_2017023_ICP8_JKB2r_F07T03a_000,ICP8/F07,840,1022,ER2HI1B_2017023_ICP8_JKB2r_F07T03a_000_0840,2.300701e+06,-8.993874e+05,111.351460,-67.544975,-234.70,960.39,1195.09,NaN,-13.48,5963.723460,-567.550741,41.048903,186.433,ICP8,ER2HI
2,ER2HI1B_2017023_ICP8_JKB2r_F07T05a_000,ICP8/F07,168,1111,ER2HI1B_2017023_ICP8_JKB2r_F07T05a_000_0168,2.311783e+06,-9.074552e+05,111.431730,-67.427400,-461.39,803.03,1264.42,-61.86,-32.32,7788.083221,-567.550741,134.157683,178.590,ICP8,ER2HI
3,ER2HI1B_2017023_ICP8_JKB2r_F07T05a_000,ICP8/F07,504,1021,ER2HI1B_2017023_ICP8_JKB2r_F07T05a_000_0504,2.309200e+06,-9.009693e+05,111.313995,-67.469670,-309.08,815.82,1124.90,-48.56,-13.35,6817.404817,-567.550972,95.571549,199.023,ICP8,ER2HI
4,ER2HI1B_2017023_ICP8_JKB2r_F07T05a_000,ICP8/F07,840,1038,ER2HI1B_2017023_ICP8_JKB2r_F07T05a_000_0840,2.303299e+06,-9.004596e+05,111.352684,-67.520050,-305.99,893.33,1199.32,NaN,-29.01,5881.549013,-567.550972,65.878738,183.386,ICP8,ER2HI
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10810,IR2HI1B_2012346_WSB_JKB2h_R40b_010,ICP5/F18,504,1633,IR2HI1B_2012346_WSB_JKB2h_R40b_010_0504,5.253390e+05,-1.209101e+06,156.515850,-77.910126,383.18,2149.82,1766.64,-70.47,-13.95,7733.431278,-567.550741,122.428389,167.868,ICP5,IR2HI
10811,IR2HI1B_2012346_WSB_JKB2h_R40b_010,ICP5/F18,840,1178,IR2HI1B_2012346_WSB_JKB2h_R40b_010_0840,5.180543e+05,-1.211759e+06,156.852360,-77.914154,1144.23,2133.79,989.56,-54.06,-12.14,7780.654742,-567.550741,234.919537,185.486,ICP5,IR2HI
10812,IR2HI1B_2012346_WSB_JKB2h_R40b_010,ICP5/F18,1176,1643,IR2HI1B_2012346_WSB_JKB2h_R40b_010_1176,5.107318e+05,-1.214441e+06,157.191020,-77.917700,362.83,2150.23,1787.40,NaN,-14.13,7810.907713,-567.550741,242.606880,149.676,ICP5,IR2HI
10813,IR2HI1B_2012346_WSB_JKB2h_R40b_010,ICP5/F18,1512,1134,IR2HI1B_2012346_WSB_JKB2h_R40b_010_1512,5.033827e+05,-1.217118e+06,157.530870,-77.920960,1257.48,2227.37,969.89,-37.47,-14.09,7836.124150,-567.550895,264.692946,198.221,ICP5,IR2HI


## Calculate the pressure melting point (pmp)

In [3]:
# Clausius-Clapeyron slope for ice (K/Pa), air-saturated water
beta = 9.8e-8
# ice density (kg/m^3)
rho = 910
# gravitational acceleration (m/s^2)
g = 9.81
# pressure melting point (deg C); ice_thickness in metres
meta["pmp"] = - beta * rho * g * meta["ice_thickness"]

print(meta["pmp"].min(), meta["pmp"].max())

-3.8036368073340006 -0.24789039093000007


## Add ice velocity

In [4]:
ice_vel = xr.open_dataset(path_to_phase_based_ice_vel)

In [5]:
ice_vel_on_target = ice_vel.interp(
    x= xr.DataArray(meta["X"].values, dims = "points"),
    y = xr.DataArray(meta["Y"].values, dims = "points"),
    # linear is a good compromise of slow cubic and heurstic nearest
    method = "linear",
)

In [6]:
# Add three variable
meta["VY"] = ice_vel_on_target["VY"]
meta["VX"] = ice_vel_on_target["VX"]
meta["ice_speed"] = np.hypot(ice_vel_on_target["VX"], ice_vel_on_target["VY"])

## Add basal friction coefficient

In [7]:
beta = xr.open_dataset(path_to_beta_basal_friction)

In [8]:
# confirming x and y on PS
beta

<xarray.Dataset> Size: 92MB
Dimensions:  (y: 4380, x: 5256)
Coordinates:
  * y        (y) float64 35kB -2.14e+06 -2.139e+06 ... 2.238e+06 2.239e+06
  * x        (x) float64 42kB -2.51e+06 -2.509e+06 ... 2.744e+06 2.745e+06
Data variables:
    beta     (y, x) float32 92MB ...
Attributes: (12/13)
    title:                       beta grid for Elmer/Ice GridDataReader
    description:                 Data extracted from Chen Zhao's ISMIP6 2300 ...
    source:                      beta_alpha_ismip6_cz.csv
    created_by:                  convert_beta_alpha_ismip6_cz_to_nc.ipynb
    created_on:                  2026-05-29T02:04:39Z
    Conventions:                 CF-1.8
    ...                          ...
    gridding_method:             idw
    footprint_mode:              none
    footprint_dilation_cells:    2
    max_nodata_fill_distance_m:  None
    max_distance_m:              None
    post_smooth_sigma_cells:     1.0

In [9]:
basal_friction_on_target = beta.interp(
    x= xr.DataArray(meta["X"].values, dims = "points"),
    y = xr.DataArray(meta["Y"].values, dims = "points"),
    # linear is a good compromise of slow cubic and heurstic nearest
    method = "linear",
)

In [10]:
meta["basal_friction_beta"] = basal_friction_on_target["beta"]

In [18]:
label_tensor = torch.tensor(meta["basal_friction_beta"])
torch.save(label_tensor, "data/label_tensor.pt")

# Export enriched metadata

In [11]:
meta.to_csv(path_to_tiles_metadata_enriched)